In [ ]:

import pandas as pd
import numpy as np
import random
import os
import zipfile

# ----------------------------------------
# QUICKBITE - PHASE 1: DATASET GENERATION
# ----------------------------------------

random.seed(42)
np.random.seed(42)

os.makedirs("quickbite_data", exist_ok=True)

# 1. CUSTOMERS
customers = pd.DataFrame({
    "customer_id": [f"CUST{i:04d}" for i in range(1, 101)],
    "customer_name": [f"Customer_{i}" for i in range(1, 101)],
    "email": [f"customer{i}@example.com" for i in range(1, 101)],
    "city": np.random.choice(
        ["Hyderabad", "Bangalore", "Chennai", "Mumbai"],
        100
    )
})

# 2. RESTAURANTS
restaurants = pd.DataFrame({
    "restaurant_id": [f"REST{i:03d}" for i in range(1, 16)],
    "restaurant_name": [
        f"Restaurant_{i}" for i in range(1, 16)
    ],
    "cuisine": np.random.choice(
        ["Indian", "Chinese", "Italian", "Mexican", "Fast Food"],
        15
    ),
    "city": np.random.choice(
        ["Hyderabad", "Bangalore", "Chennai", "Mumbai"],
        15
    )
})

# 3. DELIVERY PARTNERS
partners = pd.DataFrame({
    "partner_id": [f"DP{i:03d}" for i in range(1, 21)],
    "partner_name": [
        f"Partner_{i}" for i in range(1, 21)
    ],
    "vehicle_type": np.random.choice(
        ["Bike", "Scooter", "Bicycle"],
        20
    )
})

# 4. ORDERS - CREATE 500 ORIGINAL ORDERS
statuses = [
    "PLACED",
    "PREPARING",
    "READY_FOR_PICKUP",
    "PICKED_UP",
    "DELIVERED",
    "CANCELLED"
]

orders = pd.DataFrame({
    "order_id": [f"ORD{i:05d}" for i in range(1, 501)],
    "customer_id": np.random.choice(customers["customer_id"], 500),
    "restaurant_id": np.random.choice(restaurants["restaurant_id"], 500),
    "partner_id": np.random.choice(partners["partner_id"], 500),
    "order_timestamp": (
        pd.Timestamp("2026-11-01")
        + pd.to_timedelta(
            np.random.randint(0, 30 * 24 * 60, 500),
            unit="m"
        )
    ),
    "order_amount": np.random.randint(100, 1501, 500),
    "payment_method": np.random.choice(
        ["UPI", "CARD", "CASH", "WALLET"],
        500
    ),
    "order_status": np.random.choice(
        statuses,
        500,
        p=[0.10, 0.15, 0.15, 0.10, 0.40, 0.10]
    )
})

# Guarantee some delivered and cancelled orders
orders.loc[0, "order_status"] = "DELIVERED"
orders.loc[1, "order_status"] = "CANCELLED"

# 5. CREATE DELIVERY EVENTS FROM ORIGINAL ORDERS
events = []

for _, order in orders.iterrows():

    order_id = order["order_id"]
    base_time = order["order_timestamp"]

    events.append({
        "event_id": f"EVT_{order_id}_01",
        "order_id": order_id,
        "event_type": "ORDER_PLACED",
        "event_timestamp": base_time
    })

    events.append({
        "event_id": f"EVT_{order_id}_02",
        "order_id": order_id,
        "event_type": "RESTAURANT_ACCEPTED",
        "event_timestamp": base_time + pd.Timedelta(minutes=5)
    })

    events.append({
        "event_id": f"EVT_{order_id}_03",
        "order_id": order_id,
        "event_type": "FOOD_READY",
        "event_timestamp": base_time + pd.Timedelta(minutes=20)
    })

    if order["order_status"] in ["PICKED_UP", "DELIVERED"]:
        events.append({
            "event_id": f"EVT_{order_id}_04",
            "order_id": order_id,
            "event_type": "PICKED_UP",
            "event_timestamp": base_time + pd.Timedelta(minutes=30)
        })

    if order["order_status"] == "DELIVERED":
        events.append({
            "event_id": f"EVT_{order_id}_05",
            "order_id": order_id,
            "event_type": "DELIVERED",
            "event_timestamp": base_time + pd.Timedelta(minutes=55)
        })

    if order["order_status"] == "CANCELLED":
        events.append({
            "event_id": f"EVT_{order_id}_06",
            "order_id": order_id,
            "event_type": "CANCELLED",
            "event_timestamp": base_time + pd.Timedelta(minutes=10)
        })

events = pd.DataFrame(events)

# ----------------------------------------
# INTRODUCE INTENTIONAL DATA QUALITY ISSUES
# ----------------------------------------

# Issue 1: Duplicate order ID
duplicate_order = orders.iloc[[2]].copy()
orders = pd.concat([orders, duplicate_order], ignore_index=True)

# Issue 2: Missing customer ID
orders.loc[len(orders)] = orders.iloc[3].copy()
orders.loc[orders.index[-1], "order_id"] = "ORD90001"
orders.loc[orders.index[-1], "customer_id"] = None

# Issue 3: Invalid restaurant ID
orders.loc[len(orders)] = orders.iloc[4].copy()
orders.loc[orders.index[-1], "order_id"] = "ORD90002"
orders.loc[orders.index[-1], "restaurant_id"] = "REST999"

# Issue 4: Zero order amount
orders.loc[len(orders)] = orders.iloc[5].copy()
orders.loc[orders.index[-1], "order_id"] = "ORD90003"
orders.loc[orders.index[-1], "order_amount"] = 0

# Issue 5: Invalid order status
orders.loc[len(orders)] = orders.iloc[6].copy()
orders.loc[orders.index[-1], "order_id"] = "ORD90004"
orders.loc[orders.index[-1], "order_status"] = "IN_TRANSIT"

# Issue 6: Invalid delivery partner reference
orders.loc[len(orders)] = orders.iloc[7].copy()
orders.loc[orders.index[-1], "order_id"] = "ORD90005"
orders.loc[orders.index[-1], "partner_id"] = "DP999"

# Issue 7: Duplicate delivery event
events = pd.concat([events, events.iloc[[0]]], ignore_index=True)

# Issue 8: Missing order ID in event
events.loc[len(events)] = [
    "EVT_BAD_01", None, "DELIVERED",
    pd.Timestamp("2026-11-15 12:00:00")
]

# Issue 9: Delivery timestamp before pickup
delivered_order = orders.iloc[0]["order_id"]

delivered_event_index = events[
    (events["order_id"] == delivered_order) &
    (events["event_type"] == "DELIVERED")
].index[0]

events.loc[delivered_event_index, "event_timestamp"] = (
    orders.iloc[0]["order_timestamp"] + pd.Timedelta(minutes=25)
)

# Issue 10: Cancelled order marked as delivered
cancelled_order = orders.iloc[1]["order_id"]

events.loc[len(events)] = [
    "EVT_BAD_02",
    cancelled_order,
    "DELIVERED",
    orders.iloc[1]["order_timestamp"] + pd.Timedelta(minutes=60)
]

# ----------------------------------------
# SAVE ALL FIVE CSV FILES
# ----------------------------------------

datasets = {
    "customers.csv": customers,
    "restaurants.csv": restaurants,
    "orders.csv": orders,
    "delivery_partners.csv": partners,
    "delivery_events.csv": events
}

for filename, dataframe in datasets.items():
    filepath = f"quickbite_data/{filename}"
    dataframe.to_csv(filepath, index=False)
    print(f"Created: {filename} | Rows: {len(dataframe)}")

# ----------------------------------------
# CREATE ZIP FILE
# ----------------------------------------

zip_path = "QuickBite_Datasets.zip"

with zipfile.ZipFile(zip_path, "w") as zip_file:
    for filename in datasets:
        zip_file.write(
            f"quickbite_data/{filename}",
            arcname=filename
        )

print("\nPhase 1 completed successfully!")
print("ZIP file created:", zip_path)

# Download ZIP to your computer
from google.colab import files
files.download(zip_path)


Created: customers.csv | Rows: 100
Created: restaurants.csv | Rows: 15
Created: orders.csv | Rows: 506
Created: delivery_partners.csv | Rows: 20
Created: delivery_events.csv | Rows: 2014

Phase 1 completed successfully!
ZIP file created: QuickBite_Datasets.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import pandas as pd

folder = "quickbite_data"

for filename in os.listdir(folder):
    df = pd.read_csv(f"{folder}/{filename}")

    print("\nFile:", filename)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    display(df.head(5))



File: delivery_partners.csv
Rows: 20
Columns: ['partner_id', 'partner_name', 'vehicle_type']


,partner_id,partner_name,vehicle_type
0,DP001,Partner_1,Bicycle
1,DP002,Partner_2,Bicycle
2,DP003,Partner_3,Bicycle
3,DP004,Partner_4,Bike
4,DP005,Partner_5,Bicycle



File: restaurants.csv
Rows: 15
Columns: ['restaurant_id', 'restaurant_name', 'cuisine', 'city']


,restaurant_id,restaurant_name,cuisine,city
0,REST001,Restaurant_1,Chinese,Mumbai
1,REST002,Restaurant_2,Chinese,Mumbai
2,REST003,Restaurant_3,Mexican,Hyderabad
3,REST004,Restaurant_4,Chinese,Hyderabad
4,REST005,Restaurant_5,Chinese,Hyderabad



File: delivery_events.csv
Rows: 2014
Columns: ['event_id', 'order_id', 'event_type', 'event_timestamp']


,event_id,order_id,event_type,event_timestamp
0,EVT_ORD00001_01,ORD00001,ORDER_PLACED,2026-11-04 02:54:00
1,EVT_ORD00001_02,ORD00001,RESTAURANT_ACCEPTED,2026-11-04 02:59:00
2,EVT_ORD00001_03,ORD00001,FOOD_READY,2026-11-04 03:14:00
3,EVT_ORD00001_04,ORD00001,PICKED_UP,2026-11-04 03:24:00
4,EVT_ORD00001_05,ORD00001,DELIVERED,2026-11-04 03:19:00



File: orders.csv
Rows: 506
Columns: ['order_id', 'customer_id', 'restaurant_id', 'partner_id', 'order_timestamp', 'order_amount', 'payment_method', 'order_status']


,order_id,customer_id,restaurant_id,partner_id,order_timestamp,order_amount,payment_method,order_status
0,ORD00001,CUST0088,REST012,DP011,2026-11-04 02:54:00,460,CARD,DELIVERED
1,ORD00002,CUST0037,REST009,DP004,2026-11-11 12:32:00,1343,UPI,CANCELLED
2,ORD00003,CUST0099,REST004,DP015,2026-11-17 14:15:00,1109,UPI,DELIVERED
3,ORD00004,CUST0044,REST011,DP006,2026-11-07 08:35:00,803,UPI,PREPARING
4,ORD00005,CUST0086,REST005,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP



File: customers.csv
Rows: 100
Columns: ['customer_id', 'customer_name', 'email', 'city']


,customer_id,customer_name,email,city
0,CUST0001,Customer_1,customer1@example.com,Chennai
1,CUST0002,Customer_2,customer2@example.com,Mumbai
2,CUST0003,Customer_3,customer3@example.com,Hyderabad
3,CUST0004,Customer_4,customer4@example.com,Chennai
4,CUST0005,Customer_5,customer5@example.com,Chennai


In [ ]:

import pandas as pd
import os
import uuid
import hashlib
from datetime import datetime, timezone

# -----------------------------------------
# QUICKBITE - PHASE 2: DATA INGESTION
# -----------------------------------------

SOURCE_FOLDER = "quickbite_data"

EXPECTED_FILES = {
    "customers.csv": [
        "customer_id",
        "customer_name",
        "email",
        "city"
    ],

    "restaurants.csv": [
        "restaurant_id",
        "restaurant_name",
        "cuisine",
        "city"
    ],

    "orders.csv": [
        "order_id",
        "customer_id",
        "restaurant_id",
        "partner_id",
        "order_timestamp",
        "order_amount",
        "payment_method",
        "order_status"
    ],

    "delivery_partners.csv": [
        "partner_id",
        "partner_name",
        "vehicle_type"
    ],

    "delivery_events.csv": [
        "event_id",
        "order_id",
        "event_type",
        "event_timestamp"
    ]
}

# Create a unique ID for this ingestion run
RUN_ID = str(uuid.uuid4())

# Record the ingestion timestamp
LOADED_AT = datetime.now(timezone.utc).isoformat()

# Storage for ingested data
raw_data = {}

# File-level ingestion report
ingestion_report = []

print("QUICKBITE DATA INGESTION STARTED")
print("Run ID:", RUN_ID)
print("Loaded at:", LOADED_AT)
print("-" * 60)

# -----------------------------------------
# CHECK AND READ SOURCE FILES
# -----------------------------------------

for filename, expected_columns in EXPECTED_FILES.items():

    filepath = os.path.join(SOURCE_FOLDER, filename)

    # Check whether file exists
    if not os.path.exists(filepath):

        print("ERROR: Missing file:", filename)

        ingestion_report.append({
            "file_name": filename,
            "status": "FAILED",
            "reason": "Source file not found"
        })

        continue

    try:
        # Read as strings to preserve raw source values
        df = pd.read_csv(filepath, dtype="string")

        # Check required columns
        missing_columns = [
            col for col in expected_columns
            if col not in df.columns
        ]

        if missing_columns:

            print("ERROR:", filename)
            print("Missing columns:", missing_columns)

            ingestion_report.append({
                "file_name": filename,
                "status": "FAILED",
                "reason": "Required columns missing"
            })

            continue

        # Preserve original column values and add metadata
        df["source_file"] = filename

        # CSV row number: header is row 1
        df["source_row_number"] = range(2, len(df) + 2)

        df["loaded_at"] = LOADED_AT
        df["run_id"] = RUN_ID

        # Generate a hash for each original source row
        original_columns = expected_columns

        df["content_hash"] = df[original_columns].apply(
            lambda row: hashlib.sha256(
                "|".join(
                    "" if pd.isna(value) else str(value)
                    for value in row
                ).encode("utf-8")
            ).hexdigest(),
            axis=1
        )

        # Store the ingested DataFrame
        raw_data[filename.replace(".csv", "")] = df

        ingestion_report.append({
            "file_name": filename,
            "status": "SUCCESS",
            "row_count": len(df),
            "column_count": len(expected_columns),
            "reason": "File ingested successfully"
        })

        print(
            f"SUCCESS: {filename} | "
            f"Rows: {len(df)} | "
            f"Columns: {len(expected_columns)}"
        )

    except Exception as e:

        print("ERROR:", filename, "|", str(e))

        ingestion_report.append({
            "file_name": filename,
            "status": "FAILED",
            "reason": str(e)
        })

# -----------------------------------------
# CREATE INGESTION REPORT
# -----------------------------------------

ingestion_report = pd.DataFrame(ingestion_report)

print("\nINGESTION SUMMARY")
print("-" * 60)

display(ingestion_report)

successful_files = (
    ingestion_report["status"] == "SUCCESS"
).sum()

failed_files = (
    ingestion_report["status"] == "FAILED"
).sum()

print("\nSuccessful files:", successful_files)
print("Failed files:", failed_files)

if failed_files == 0:
    print("\nINGESTION COMPLETED SUCCESSFULLY")
else:
    print("\nINGESTION COMPLETED WITH ERRORS")


QUICKBITE DATA INGESTION STARTED
Run ID: 7a5cd186-7ea5-4663-a08d-541e1c662793
Loaded at: 2026-11-25T11:00:49.283194+00:00
------------------------------------------------------------
SUCCESS: customers.csv | Rows: 100 | Columns: 4
SUCCESS: restaurants.csv | Rows: 15 | Columns: 4
SUCCESS: orders.csv | Rows: 506 | Columns: 8
SUCCESS: delivery_partners.csv | Rows: 20 | Columns: 3
SUCCESS: delivery_events.csv | Rows: 2014 | Columns: 4

INGESTION SUMMARY
------------------------------------------------------------


,file_name,status,row_count,column_count,reason
0,customers.csv,SUCCESS,100,4,File ingested successfully
1,restaurants.csv,SUCCESS,15,4,File ingested successfully
2,orders.csv,SUCCESS,506,8,File ingested successfully
3,delivery_partners.csv,SUCCESS,20,3,File ingested successfully
4,delivery_events.csv,SUCCESS,2014,4,File ingested successfully



Successful files: 5
Failed files: 0

INGESTION COMPLETED SUCCESSFULLY


In [ ]:

# Inspect ingested datasets

for dataset_name, df in raw_data.items():

    print("\nDataset:", dataset_name)
    print("Total rows:", len(df))
    print("Total columns:", len(df.columns))

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nFirst 3 records:")
    display(df.head(3))



Dataset: customers
Total rows: 100
Total columns: 9

Column names:
['customer_id', 'customer_name', 'email', 'city', 'source_file', 'source_row_number', 'loaded_at', 'run_id', 'content_hash']

First 3 records:


,customer_id,customer_name,email,city,source_file,source_row_number,loaded_at,run_id,content_hash
0,CUST0001,Customer_1,customer1@example.com,Chennai,customers.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,94c38737013acf3f98ba1839a8447a89c08af4dc017b44...
1,CUST0002,Customer_2,customer2@example.com,Mumbai,customers.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,ceff95991d4c7b3f482a4009c7fdc925c0710010a1009e...
2,CUST0003,Customer_3,customer3@example.com,Hyderabad,customers.csv,4,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,a5d0c8b1ed8c964e147e8218ce5cadadf4b1c06c44d32a...



Dataset: restaurants
Total rows: 15
Total columns: 9

Column names:
['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'source_file', 'source_row_number', 'loaded_at', 'run_id', 'content_hash']

First 3 records:


,restaurant_id,restaurant_name,cuisine,city,source_file,source_row_number,loaded_at,run_id,content_hash
0,REST001,Restaurant_1,Chinese,Mumbai,restaurants.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,5f1098c000c01826fe7e7aa517a9ebfed4111ce8e8a25d...
1,REST002,Restaurant_2,Chinese,Mumbai,restaurants.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,ceb933b28f8b542cc27eaa96587e8f67401b35e11f95e8...
2,REST003,Restaurant_3,Mexican,Hyderabad,restaurants.csv,4,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,af762aae9dcbc1da93e93a7d61a46224b9dbf1b213add7...



Dataset: orders
Total rows: 506
Total columns: 13

Column names:
['order_id', 'customer_id', 'restaurant_id', 'partner_id', 'order_timestamp', 'order_amount', 'payment_method', 'order_status', 'source_file', 'source_row_number', 'loaded_at', 'run_id', 'content_hash']

First 3 records:


,order_id,customer_id,restaurant_id,partner_id,order_timestamp,order_amount,payment_method,order_status,source_file,source_row_number,loaded_at,run_id,content_hash
0,ORD00001,CUST0088,REST012,DP011,2026-11-04 02:54:00,460,CARD,DELIVERED,orders.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,0390f6eda5bda61a13818ac9c5f2574e3fb8c40f2e3d5a...
1,ORD00002,CUST0037,REST009,DP004,2026-11-11 12:32:00,1343,UPI,CANCELLED,orders.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,4415ebf33f1173222e7c39ab73a68bc362bf80299961da...
2,ORD00003,CUST0099,REST004,DP015,2026-11-17 14:15:00,1109,UPI,DELIVERED,orders.csv,4,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,7a4cccdfd06444cf12fa6d26b033ba83d7daff79be1d87...



Dataset: delivery_partners
Total rows: 20
Total columns: 8

Column names:
['partner_id', 'partner_name', 'vehicle_type', 'source_file', 'source_row_number', 'loaded_at', 'run_id', 'content_hash']

First 3 records:


,partner_id,partner_name,vehicle_type,source_file,source_row_number,loaded_at,run_id,content_hash
0,DP001,Partner_1,Bicycle,delivery_partners.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,cfbfddf1b00fec5b23c66d3a6bca0a9a06eedb922f128d...
1,DP002,Partner_2,Bicycle,delivery_partners.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,d2e8c63f682275ee74bf9fd8686d082e5e1a66442ab63f...
2,DP003,Partner_3,Bicycle,delivery_partners.csv,4,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,03d1202706816a06a0db41d3fbd9726ca26670a1b114e6...



Dataset: delivery_events
Total rows: 2014
Total columns: 9

Column names:
['event_id', 'order_id', 'event_type', 'event_timestamp', 'source_file', 'source_row_number', 'loaded_at', 'run_id', 'content_hash']

First 3 records:


,event_id,order_id,event_type,event_timestamp,source_file,source_row_number,loaded_at,run_id,content_hash
0,EVT_ORD00001_01,ORD00001,ORDER_PLACED,2026-11-04 02:54:00,delivery_events.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,54485cf1743bc147a01a1701e6a198a9ffa2ca9d8fb4a1...
1,EVT_ORD00001_02,ORD00001,RESTAURANT_ACCEPTED,2026-11-04 02:59:00,delivery_events.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,560965b56eb550a56b3aa33dd802a9d84d7dbcd2a40a13...
2,EVT_ORD00001_03,ORD00001,FOOD_READY,2026-11-04 03:14:00,delivery_events.csv,4,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,d4790f67444f33d4f39662b8fbde8ab212be4f9434e46d...


In [ ]:

# Save ingested raw datasets

os.makedirs("quickbite_raw", exist_ok=True)

for dataset_name, df in raw_data.items():

    output_path = f"quickbite_raw/{dataset_name}.csv"

    df.to_csv(output_path, index=False)

    print("Saved:", output_path)

# Save ingestion report
ingestion_report.to_csv(
    "quickbite_raw/ingestion_report.csv",
    index=False
)

print("\nRaw ingestion data saved successfully!")


Saved: quickbite_raw/customers.csv
Saved: quickbite_raw/restaurants.csv
Saved: quickbite_raw/orders.csv
Saved: quickbite_raw/delivery_partners.csv
Saved: quickbite_raw/delivery_events.csv

Raw ingestion data saved successfully!


In [ ]:

# -----------------------------------------
# QUICKBITE - PHASE 3: ORDER VALIDATION
# -----------------------------------------

import pandas as pd
import numpy as np
import os

# Work with a copy of ingested data
orders_check = raw_data["orders"].copy()

# Valid business statuses
VALID_ORDER_STATUSES = {
    "PLACED",
    "PREPARING",
    "READY_FOR_PICKUP",
    "PICKED_UP",
    "DELIVERED",
    "CANCELLED"
}

# Reference data
valid_customers = set(
    raw_data["customers"]["customer_id"].dropna()
)

valid_restaurants = set(
    raw_data["restaurants"]["restaurant_id"].dropna()
)

valid_partners = set(
    raw_data["delivery_partners"]["partner_id"].dropna()
)

# Prepare validation columns
orders_check["validation_errors"] = ""
orders_check["pipeline_status"] = "PUBLISH"

# Helper function to record validation failures
def add_order_error(mask, rule_id, message):

    mask = mask.fillna(False)

    orders_check.loc[mask, "validation_errors"] = (
        orders_check.loc[mask, "validation_errors"]
        .apply(
            lambda x: f"{x}; {rule_id}: {message}"
            if x else f"{rule_id}: {message}"
        )
    )

    orders_check.loc[mask, "pipeline_status"] = "REJECT"


# QB-001: Order ID must not be null
add_order_error(
    orders_check["order_id"].isna() |
    orders_check["order_id"].str.strip().eq(""),
    "QB-001",
    "Missing order ID"
)

# QB-002: Order ID must be unique
duplicate_mask = (
    orders_check["order_id"].notna() &
    orders_check["order_id"].duplicated(keep=False)
)

add_order_error(
    duplicate_mask,
    "QB-002",
    "Duplicate order ID"
)

# QB-003: Customer ID must exist
add_order_error(
    ~orders_check["customer_id"].isin(valid_customers),
    "QB-003",
    "Customer ID missing or not found"
)

# QB-004: Restaurant ID must exist
add_order_error(
    ~orders_check["restaurant_id"].isin(valid_restaurants),
    "QB-004",
    "Restaurant ID missing or not found"
)

# QB-005: Order amount must be greater than zero
amount_numeric = pd.to_numeric(
    orders_check["order_amount"],
    errors="coerce"
)

add_order_error(
    amount_numeric.isna() | (amount_numeric <= 0),
    "QB-005",
    "Order amount must be greater than zero"
)

# QB-006: Order status must be valid
add_order_error(
    ~orders_check["order_status"].isin(VALID_ORDER_STATUSES),
    "QB-006",
    "Invalid order status"
)

# QB-007: Partner ID must exist
add_order_error(
    ~orders_check["partner_id"].isin(valid_partners),
    "QB-007",
    "Partner ID missing or not found"
)

# Convert amount to numeric in a separate derived column
orders_check["order_amount_numeric"] = amount_numeric

print("ORDER VALIDATION COMPLETED")

display(
    orders_check[
        [
            "order_id",
            "customer_id",
            "restaurant_id",
            "partner_id",
            "order_amount",
            "order_status",
            "pipeline_status",
            "validation_errors"
        ]
    ].head(15)
)


ORDER VALIDATION COMPLETED


,order_id,customer_id,restaurant_id,partner_id,order_amount,order_status,pipeline_status,validation_errors
0,ORD00001,CUST0088,REST012,DP011,460,DELIVERED,PUBLISH,
1,ORD00002,CUST0037,REST009,DP004,1343,CANCELLED,PUBLISH,
2,ORD00003,CUST0099,REST004,DP015,1109,DELIVERED,REJECT,QB-002: Duplicate order ID
3,ORD00004,CUST0044,REST011,DP006,803,PREPARING,PUBLISH,
4,ORD00005,CUST0086,REST005,DP017,674,PICKED_UP,PUBLISH,
5,ORD00006,CUST0091,REST004,DP006,1284,CANCELLED,PUBLISH,
6,ORD00007,CUST0035,REST005,DP019,1212,DELIVERED,PUBLISH,
7,ORD00008,CUST0065,REST007,DP002,787,PLACED,PUBLISH,
8,ORD00009,CUST0099,REST009,DP015,880,PLACED,PUBLISH,
9,ORD00010,CUST0047,REST007,DP011,684,PICKED_UP,PUBLISH,


In [ ]:

# -----------------------------------------
# DELIVERY EVENT VALIDATION
# -----------------------------------------

events_check = raw_data["delivery_events"].copy()

VALID_EVENT_TYPES = {
    "ORDER_PLACED",
    "RESTAURANT_ACCEPTED",
    "FOOD_READY",
    "PICKED_UP",
    "DELIVERED",
    "CANCELLED"
}

events_check["validation_errors"] = ""
events_check["pipeline_status"] = "PUBLISH"

def add_event_error(mask, rule_id, message, status="REJECT"):

    mask = mask.fillna(False)

    events_check.loc[mask, "validation_errors"] = (
        events_check.loc[mask, "validation_errors"]
        .apply(
            lambda x: f"{x}; {rule_id}: {message}"
            if x else f"{rule_id}: {message}"
        )
    )

    # Critical errors take priority over human review
    if status == "REJECT":
        events_check.loc[mask, "pipeline_status"] = "REJECT"

    else:
        publish_mask = (
            mask &
            events_check["pipeline_status"].eq("PUBLISH")
        )
        events_check.loc[
            publish_mask, "pipeline_status"
        ] = "HUMAN_REVIEW"


# QB-008: Event ID must be unique
duplicate_event_mask = (
    events_check["event_id"].notna() &
    events_check["event_id"].duplicated(keep=False)
)

add_event_error(
    duplicate_event_mask,
    "QB-008",
    "Duplicate event ID"
)

# QB-009: Event must reference an existing order
valid_order_ids = set(
    orders_check["order_id"].dropna()
)

add_event_error(
    ~events_check["order_id"].isin(valid_order_ids),
    "QB-009",
    "Missing or unknown order ID"
)

# QB-011: Event type must be valid
add_event_error(
    ~events_check["event_type"].isin(VALID_EVENT_TYPES),
    "QB-011",
    "Invalid event type"
)

# Parse timestamps for analysis without changing raw_data
events_check["event_timestamp_parsed"] = pd.to_datetime(
    events_check["event_timestamp"],
    errors="coerce"
)

# Additional timestamp quality check
add_event_error(
    events_check["event_timestamp_parsed"].isna(),
    "QB-013",
    "Invalid event timestamp"
)

# QB-010: Delivered event must occur after pickup
pickup_times = (
    events_check[
        events_check["event_type"] == "PICKED_UP"
    ]
    .groupby("order_id")["event_timestamp_parsed"]
    .min()
)

delivered_mask = events_check["event_type"].eq("DELIVERED")

for idx in events_check[delivered_mask].index:

    order_id = events_check.loc[idx, "order_id"]
    delivery_time = events_check.loc[
        idx, "event_timestamp_parsed"
    ]

    if order_id in pickup_times.index:

        pickup_time = pickup_times.loc[order_id]

        if (
            pd.notna(delivery_time) and
            pd.notna(pickup_time) and
            delivery_time < pickup_time
        ):

            add_event_error(
                pd.Series(
                    events_check.index == idx,
                    index=events_check.index
                ),
                "QB-010",
                "Delivery occurred before pickup",
                status="HUMAN_REVIEW"
            )

# QB-012: Cancelled order must not have delivered event
cancelled_order_ids = set(
    orders_check.loc[
        orders_check["order_status"] == "CANCELLED",
        "order_id"
    ].dropna()
)

cancelled_delivered_mask = (
    events_check["order_id"].isin(cancelled_order_ids) &
    events_check["event_type"].eq("DELIVERED")
)

add_event_error(
    cancelled_delivered_mask,
    "QB-012",
    "Cancelled order has a delivered event",
    status="HUMAN_REVIEW"
)

print("DELIVERY EVENT VALIDATION COMPLETED")

display(
    events_check[
        [
            "event_id",
            "order_id",
            "event_type",
            "event_timestamp",
            "pipeline_status",
            "validation_errors"
        ]
    ].head(15)
)


DELIVERY EVENT VALIDATION COMPLETED


,event_id,order_id,event_type,event_timestamp,pipeline_status,validation_errors
0,EVT_ORD00001_01,ORD00001,ORDER_PLACED,2026-11-04 02:54:00,REJECT,QB-008: Duplicate event ID
1,EVT_ORD00001_02,ORD00001,RESTAURANT_ACCEPTED,2026-11-04 02:59:00,PUBLISH,
2,EVT_ORD00001_03,ORD00001,FOOD_READY,2026-11-04 03:14:00,PUBLISH,
3,EVT_ORD00001_04,ORD00001,PICKED_UP,2026-11-04 03:24:00,PUBLISH,
4,EVT_ORD00001_05,ORD00001,DELIVERED,2026-11-04 03:19:00,HUMAN_REVIEW,QB-010: Delivery occurred before pickup
5,EVT_ORD00002_01,ORD00002,ORDER_PLACED,2026-11-11 12:32:00,PUBLISH,
6,EVT_ORD00002_02,ORD00002,RESTAURANT_ACCEPTED,2026-11-11 12:37:00,PUBLISH,
7,EVT_ORD00002_03,ORD00002,FOOD_READY,2026-11-11 12:52:00,PUBLISH,
8,EVT_ORD00002_06,ORD00002,CANCELLED,2026-11-11 12:42:00,PUBLISH,
9,EVT_ORD00003_01,ORD00003,ORDER_PLACED,2026-11-17 14:15:00,PUBLISH,


In [ ]:

# -----------------------------------------
# CREATE EXCEPTION REPORT
# -----------------------------------------

order_exceptions = orders_check[
    orders_check["pipeline_status"] != "PUBLISH"
].copy()

event_exceptions = events_check[
    events_check["pipeline_status"] != "PUBLISH"
].copy()

# Standardize report columns
order_exceptions["source_dataset"] = "orders"

event_exceptions["source_dataset"] = "delivery_events"

# Combine exceptions
exception_report = pd.concat(
    [order_exceptions, event_exceptions],
    ignore_index=True,
    sort=False
)

print("EXCEPTION REPORT CREATED")

print("Order exceptions:", len(order_exceptions))
print("Event exceptions:", len(event_exceptions))
print("Total exceptions:", len(exception_report))

display(
    exception_report[
        [
            "source_dataset",
            "pipeline_status",
            "validation_errors"
        ]
    ].head(20)
)

# Save exception report
os.makedirs("quickbite_validation", exist_ok=True)

exception_report.to_csv(
    "quickbite_validation/exception_report.csv",
    index=False
)

print("\nException report saved successfully!")


EXCEPTION REPORT CREATED
Order exceptions: 7
Event exceptions: 5
Total exceptions: 12


,source_dataset,pipeline_status,validation_errors
0,orders,REJECT,QB-002: Duplicate order ID
1,orders,REJECT,QB-002: Duplicate order ID
2,orders,REJECT,QB-003: Customer ID missing or not found
3,orders,REJECT,QB-004: Restaurant ID missing or not found
4,orders,REJECT,QB-005: Order amount must be greater than zero
5,orders,REJECT,QB-006: Invalid order status
6,orders,REJECT,QB-007: Partner ID missing or not found
7,delivery_events,REJECT,QB-008: Duplicate event ID
8,delivery_events,HUMAN_REVIEW,QB-010: Delivery occurred before pickup
9,delivery_events,REJECT,QB-008: Duplicate event ID



Exception report saved successfully!


In [ ]:

# -----------------------------------------
# VALIDATION SUMMARY
# -----------------------------------------

order_summary = (
    orders_check["pipeline_status"]
    .value_counts()
    .rename_axis("pipeline_status")
    .reset_index(name="record_count")
)

event_summary = (
    events_check["pipeline_status"]
    .value_counts()
    .rename_axis("pipeline_status")
    .reset_index(name="record_count")
)

print("ORDER VALIDATION SUMMARY")
display(order_summary)

print("DELIVERY EVENT VALIDATION SUMMARY")
display(event_summary)

# Overall summary
overall_summary = pd.DataFrame({
    "dataset": ["orders", "delivery_events"],
    "total_records": [
        len(orders_check),
        len(events_check)
    ],
    "publish": [
        (orders_check["pipeline_status"] == "PUBLISH").sum(),
        (events_check["pipeline_status"] == "PUBLISH").sum()
    ],
    "reject": [
        (orders_check["pipeline_status"] == "REJECT").sum(),
        (events_check["pipeline_status"] == "REJECT").sum()
    ],
    "human_review": [
        (orders_check["pipeline_status"] == "HUMAN_REVIEW").sum(),
        (events_check["pipeline_status"] == "HUMAN_REVIEW").sum()
    ]
})

print("OVERALL VALIDATION REPORT")
display(overall_summary)

overall_summary.to_csv(
    "quickbite_validation/validation_summary.csv",
    index=False
)


ORDER VALIDATION SUMMARY


,pipeline_status,record_count
0,PUBLISH,499
1,REJECT,7


DELIVERY EVENT VALIDATION SUMMARY


,pipeline_status,record_count
0,PUBLISH,2009
1,REJECT,3
2,HUMAN_REVIEW,2


OVERALL VALIDATION REPORT


,dataset,total_records,publish,reject,human_review
0,orders,506,499,7,0
1,delivery_events,2014,2009,3,2


In [ ]:

# -----------------------------------------
# QUICKBITE - PHASE 4: ORDER TRANSFORMATION
# -----------------------------------------

import pandas as pd
import numpy as np
import os

os.makedirs("quickbite_transformed", exist_ok=True)

# Work with a copy of validated orders
orders_transformed = orders_check.copy()

# Normalize text fields
text_columns = [
    "order_id",
    "customer_id",
    "restaurant_id",
    "partner_id",
    "payment_method",
    "order_status"
]

for col in text_columns:
    orders_transformed[col] = (
        orders_transformed[col]
        .str.strip()
    )

# Standardize categorical values
orders_transformed["order_status"] = (
    orders_transformed["order_status"].str.upper()
)

orders_transformed["payment_method"] = (
    orders_transformed["payment_method"].str.upper()
)

# Convert order amount to numeric
orders_transformed["order_amount"] = pd.to_numeric(
    orders_transformed["order_amount"],
    errors="coerce"
)

# Convert order timestamp to datetime
orders_transformed["order_timestamp"] = pd.to_datetime(
    orders_transformed["order_timestamp"],
    errors="coerce"
)

# Identify transformation errors
invalid_timestamp = (
    orders_transformed["order_timestamp"].isna()
)

# Add transformation error details
orders_transformed["transformation_errors"] = ""

orders_transformed.loc[
    invalid_timestamp,
    "transformation_errors"
] = "QB-015: Invalid order timestamp"

# Invalid timestamps cannot be trusted
orders_transformed.loc[
    invalid_timestamp,
    "pipeline_status"
] = "REJECT"

# Trusted orders
trusted_orders = orders_transformed[
    orders_transformed["pipeline_status"] == "PUBLISH"
].copy()

# Records excluded during transformation
order_rejects = orders_transformed[
    orders_transformed["pipeline_status"] != "PUBLISH"
].copy()

print("ORDER TRANSFORMATION COMPLETED")
print("Total orders:", len(orders_transformed))
print("Trusted orders:", len(trusted_orders))
print("Rejected orders:", len(order_rejects))

display(trusted_orders.head())


ORDER TRANSFORMATION COMPLETED
Total orders: 506
Trusted orders: 499
Rejected orders: 7


,order_id,customer_id,restaurant_id,partner_id,order_timestamp,order_amount,payment_method,order_status,source_file,source_row_number,loaded_at,run_id,content_hash,validation_errors,pipeline_status,order_amount_numeric,transformation_errors
0,ORD00001,CUST0088,REST012,DP011,2026-11-04 02:54:00,460,CARD,DELIVERED,orders.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,0390f6eda5bda61a13818ac9c5f2574e3fb8c40f2e3d5a...,,PUBLISH,460,
1,ORD00002,CUST0037,REST009,DP004,2026-11-11 12:32:00,1343,UPI,CANCELLED,orders.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,4415ebf33f1173222e7c39ab73a68bc362bf80299961da...,,PUBLISH,1343,
3,ORD00004,CUST0044,REST011,DP006,2026-11-07 08:35:00,803,UPI,PREPARING,orders.csv,5,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,8fbb71d17449f8ee84ed266d789805c5030fa5e1ae2939...,,PUBLISH,803,
4,ORD00005,CUST0086,REST005,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP,orders.csv,6,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,c3db8e197910fa66471003689d737a6585d2c23a5eaf3a...,,PUBLISH,674,
5,ORD00006,CUST0091,REST004,DP006,2026-11-30 23:54:00,1284,CASH,CANCELLED,orders.csv,7,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,002c456f37eee665ff52b36dba9f8ba8c6ca3b72d1d9ce...,,PUBLISH,1284,


In [ ]:

# -----------------------------------------
# DELIVERY EVENT TRANSFORMATION
# -----------------------------------------

events_transformed = events_check.copy()

# Standardize event values
for col in ["event_id", "order_id", "event_type"]:
    events_transformed[col] = (
        events_transformed[col].str.strip()
    )

events_transformed["event_type"] = (
    events_transformed["event_type"].str.upper()
)

# Convert timestamp
events_transformed["event_timestamp"] = pd.to_datetime(
    events_transformed["event_timestamp"],
    errors="coerce"
)

# Identify events connected to trusted orders
trusted_order_ids = set(
    trusted_orders["order_id"].dropna()
)

invalid_parent_mask = (
    events_transformed["pipeline_status"].eq("PUBLISH") &
    ~events_transformed["order_id"].isin(trusted_order_ids)
)

# These events cannot be published independently
events_transformed.loc[
    invalid_parent_mask,
    "pipeline_status"
] = "REJECT"

events_transformed.loc[
    invalid_parent_mask,
    "validation_errors"
] = (
    events_transformed.loc[
        invalid_parent_mask, "validation_errors"
    ].apply(
        lambda x: (
            f"{x}; QB-014: Parent order is not trusted"
            if x else "QB-014: Parent order is not trusted"
        )
    )
)

# Trusted events must have trusted parent orders
trusted_events = events_transformed[
    events_transformed["pipeline_status"].eq("PUBLISH") &
    events_transformed["order_id"].isin(trusted_order_ids)
].copy()

# Transformation rejects
event_rejects = events_transformed[
    events_transformed["pipeline_status"] != "PUBLISH"
].copy()

print("DELIVERY EVENT TRANSFORMATION COMPLETED")
print("Total events:", len(events_transformed))
print("Trusted events:", len(trusted_events))
print("Rejected/review events:", len(event_rejects))

display(trusted_events.head())


DELIVERY EVENT TRANSFORMATION COMPLETED
Total events: 2014
Trusted events: 2004
Rejected/review events: 10


,event_id,order_id,event_type,event_timestamp,source_file,source_row_number,loaded_at,run_id,content_hash,validation_errors,pipeline_status,event_timestamp_parsed
1,EVT_ORD00001_02,ORD00001,RESTAURANT_ACCEPTED,2026-11-04 02:59:00,delivery_events.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,560965b56eb550a56b3aa33dd802a9d84d7dbcd2a40a13...,,PUBLISH,2026-11-04 02:59:00
2,EVT_ORD00001_03,ORD00001,FOOD_READY,2026-11-04 03:14:00,delivery_events.csv,4,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,d4790f67444f33d4f39662b8fbde8ab212be4f9434e46d...,,PUBLISH,2026-11-04 03:14:00
3,EVT_ORD00001_04,ORD00001,PICKED_UP,2026-11-04 03:24:00,delivery_events.csv,5,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,4ecebaefab48f474611fda65ea5ec0cf86d906caec3f21...,,PUBLISH,2026-11-04 03:24:00
5,EVT_ORD00002_01,ORD00002,ORDER_PLACED,2026-11-11 12:32:00,delivery_events.csv,7,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,07e5b1a660a3f9d13cb46590fc167a1bfffd1920558058...,,PUBLISH,2026-11-11 12:32:00
6,EVT_ORD00002_02,ORD00002,RESTAURANT_ACCEPTED,2026-11-11 12:37:00,delivery_events.csv,8,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,4fbb4b6e9c6c3edb4fac23b1abc2694ddc90b994893bed...,,PUBLISH,2026-11-11 12:37:00


In [ ]:

# -----------------------------------------
# BUILD DELIVERY PERFORMANCE TABLE
# -----------------------------------------

# Extract important event timestamps
event_times = (
    trusted_events
    .pivot_table(
        index="order_id",
        columns="event_type",
        values="event_timestamp",
        aggfunc="min"
    )
    .reset_index()
)

# Remove column index name
event_times.columns.name = None

# Merge orders with event timestamps
delivery_performance = trusted_orders.merge(
    event_times,
    on="order_id",
    how="left"
)

# Add restaurant details
delivery_performance = delivery_performance.merge(
    raw_data["restaurants"][
        ["restaurant_id", "restaurant_name", "cuisine"]
    ],
    on="restaurant_id",
    how="left",
    validate="many_to_one"
)

# Add customer city
delivery_performance = delivery_performance.merge(
    raw_data["customers"][
        ["customer_id", "city"]
    ].rename(columns={"city": "customer_city"}),
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Calculate preparation time
delivery_performance["preparation_minutes"] = (
    delivery_performance["FOOD_READY"] -
    delivery_performance["RESTAURANT_ACCEPTED"]
).dt.total_seconds() / 60

# Calculate delivery duration
delivery_performance["delivery_minutes"] = (
    delivery_performance["DELIVERED"] -
    delivery_performance["PICKED_UP"]
).dt.total_seconds() / 60

# Calculate total time from order to delivery
delivery_performance["total_fulfillment_minutes"] = (
    delivery_performance["DELIVERED"] -
    delivery_performance["order_timestamp"]
).dt.total_seconds() / 60

# Select useful business columns
delivery_performance = delivery_performance[
    [
        "order_id",
        "customer_id",
        "customer_city",
        "restaurant_id",
        "restaurant_name",
        "cuisine",
        "partner_id",
        "order_timestamp",
        "order_amount",
        "payment_method",
        "order_status",
        "RESTAURANT_ACCEPTED",
        "FOOD_READY",
        "PICKED_UP",
        "DELIVERED",
        "preparation_minutes",
        "delivery_minutes",
        "total_fulfillment_minutes"
    ]
]

print("DELIVERY PERFORMANCE TABLE CREATED")

display(delivery_performance.head(10))


DELIVERY PERFORMANCE TABLE CREATED


,order_id,customer_id,customer_city,restaurant_id,restaurant_name,cuisine,partner_id,order_timestamp,order_amount,payment_method,order_status,RESTAURANT_ACCEPTED,FOOD_READY,PICKED_UP,DELIVERED,preparation_minutes,delivery_minutes,total_fulfillment_minutes
0,ORD00001,CUST0088,Hyderabad,REST012,Restaurant_12,Fast Food,DP011,2026-11-04 02:54:00,460,CARD,DELIVERED,2026-11-04 02:59:00,2026-11-04 03:14:00,2026-11-04 03:24:00,NaT,15.0,NaN,NaN
1,ORD00002,CUST0037,Mumbai,REST009,Restaurant_9,Fast Food,DP004,2026-11-11 12:32:00,1343,UPI,CANCELLED,2026-11-11 12:37:00,2026-11-11 12:52:00,NaT,NaT,15.0,NaN,NaN
2,ORD00004,CUST0044,Mumbai,REST011,Restaurant_11,Chinese,DP006,2026-11-07 08:35:00,803,UPI,PREPARING,2026-11-07 08:40:00,2026-11-07 08:55:00,NaT,NaT,15.0,NaN,NaN
3,ORD00005,CUST0086,Chennai,REST005,Restaurant_5,Chinese,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP,2026-11-23 03:28:00,2026-11-23 03:43:00,2026-11-23 03:53:00,NaT,15.0,NaN,NaN
4,ORD00006,CUST0091,Mumbai,REST004,Restaurant_4,Chinese,DP006,2026-11-30 23:54:00,1284,CASH,CANCELLED,2026-11-30 23:59:00,2026-12-01 00:14:00,NaT,NaT,15.0,NaN,NaN
5,ORD00007,CUST0035,Bangalore,REST005,Restaurant_5,Chinese,DP019,2026-11-30 20:41:00,1212,CARD,DELIVERED,2026-11-30 20:46:00,2026-11-30 21:01:00,2026-11-30 21:11:00,2026-11-30 21:36:00,15.0,25.0,55.0
6,ORD00008,CUST0065,Chennai,REST007,Restaurant_7,Mexican,DP002,2026-11-02 06:28:00,787,CASH,PLACED,2026-11-02 06:33:00,2026-11-02 06:48:00,NaT,NaT,15.0,NaN,NaN
7,ORD00009,CUST0099,Bangalore,REST009,Restaurant_9,Fast Food,DP015,2026-11-30 02:18:00,880,CARD,PLACED,2026-11-30 02:23:00,2026-11-30 02:38:00,NaT,NaT,15.0,NaN,NaN
8,ORD00010,CUST0047,Mumbai,REST007,Restaurant_7,Mexican,DP011,2026-11-23 07:23:00,684,CARD,PICKED_UP,2026-11-23 07:28:00,2026-11-23 07:43:00,2026-11-23 07:53:00,NaT,15.0,NaN,NaN
9,ORD00011,CUST0078,Mumbai,REST013,Restaurant_13,Chinese,DP008,2026-11-27 02:13:00,688,UPI,CANCELLED,2026-11-27 02:18:00,2026-11-27 02:33:00,NaT,NaT,15.0,NaN,NaN


In [ ]:

# -----------------------------------------
# SAVE TRANSFORMED DATASETS
# -----------------------------------------

# Save trusted orders
trusted_orders.to_csv(
    "quickbite_transformed/trusted_orders.csv",
    index=False
)

# Save trusted delivery events
trusted_events.to_csv(
    "quickbite_transformed/trusted_delivery_events.csv",
    index=False
)

# Save delivery performance
delivery_performance.to_csv(
    "quickbite_transformed/delivery_performance.csv",
    index=False
)

# Save transformation rejects
all_rejects = pd.concat(
    [order_rejects, event_rejects],
    ignore_index=True,
    sort=False
)

all_rejects.to_csv(
    "quickbite_transformed/transformation_rejects.csv",
    index=False
)

print("ALL TRANSFORMED DATASETS SAVED")
print("--------------------------------")

for filename in os.listdir("quickbite_transformed"):
    print(filename)

print("\nTransformation completed!")


ALL TRANSFORMED DATASETS SAVED
--------------------------------
transformation_rejects.csv
trusted_delivery_events.csv
trusted_orders.csv
delivery_performance.csv

Transformation completed!


In [ ]:

# -----------------------------------------
# TRANSFORMATION SUMMARY
# -----------------------------------------

transformation_summary = pd.DataFrame({
    "dataset": [
        "Trusted Orders",
        "Trusted Delivery Events",
        "Delivery Performance",
        "Transformation Rejects"
    ],
    "record_count": [
        len(trusted_orders),
        len(trusted_events),
        len(delivery_performance),
        len(all_rejects)
    ]
})

display(transformation_summary)

print("Trusted order sample:")
display(trusted_orders.head(5))

print("Delivery performance sample:")
display(delivery_performance.head(5))

print("Rejected record sample:")
display(all_rejects.head(5))


,dataset,record_count
0,Trusted Orders,499
1,Trusted Delivery Events,2004
2,Delivery Performance,499
3,Transformation Rejects,17


Trusted order sample:


,order_id,customer_id,restaurant_id,partner_id,order_timestamp,order_amount,payment_method,order_status,source_file,source_row_number,loaded_at,run_id,content_hash,validation_errors,pipeline_status,order_amount_numeric,transformation_errors
0,ORD00001,CUST0088,REST012,DP011,2026-11-04 02:54:00,460,CARD,DELIVERED,orders.csv,2,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,0390f6eda5bda61a13818ac9c5f2574e3fb8c40f2e3d5a...,,PUBLISH,460,
1,ORD00002,CUST0037,REST009,DP004,2026-11-11 12:32:00,1343,UPI,CANCELLED,orders.csv,3,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,4415ebf33f1173222e7c39ab73a68bc362bf80299961da...,,PUBLISH,1343,
3,ORD00004,CUST0044,REST011,DP006,2026-11-07 08:35:00,803,UPI,PREPARING,orders.csv,5,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,8fbb71d17449f8ee84ed266d789805c5030fa5e1ae2939...,,PUBLISH,803,
4,ORD00005,CUST0086,REST005,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP,orders.csv,6,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,c3db8e197910fa66471003689d737a6585d2c23a5eaf3a...,,PUBLISH,674,
5,ORD00006,CUST0091,REST004,DP006,2026-11-30 23:54:00,1284,CASH,CANCELLED,orders.csv,7,2026-11-25T11:00:49.283194+00:00,7a5cd186-7ea5-4663-a08d-541e1c662793,002c456f37eee665ff52b36dba9f8ba8c6ca3b72d1d9ce...,,PUBLISH,1284,


Delivery performance sample:


,order_id,customer_id,customer_city,restaurant_id,restaurant_name,cuisine,partner_id,order_timestamp,order_amount,payment_method,order_status,RESTAURANT_ACCEPTED,FOOD_READY,PICKED_UP,DELIVERED,preparation_minutes,delivery_minutes,total_fulfillment_minutes
0,ORD00001,CUST0088,Hyderabad,REST012,Restaurant_12,Fast Food,DP011,2026-11-04 02:54:00,460,CARD,DELIVERED,2026-11-04 02:59:00,2026-11-04 03:14:00,2026-11-04 03:24:00,NaT,15.0,NaN,NaN
1,ORD00002,CUST0037,Mumbai,REST009,Restaurant_9,Fast Food,DP004,2026-11-11 12:32:00,1343,UPI,CANCELLED,2026-11-11 12:37:00,2026-11-11 12:52:00,NaT,NaT,15.0,NaN,NaN
2,ORD00004,CUST0044,Mumbai,REST011,Restaurant_11,Chinese,DP006,2026-11-07 08:35:00,803,UPI,PREPARING,2026-11-07 08:40:00,2026-11-07 08:55:00,NaT,NaT,15.0,NaN,NaN
3,ORD00005,CUST0086,Chennai,REST005,Restaurant_5,Chinese,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP,2026-11-23 03:28:00,2026-11-23 03:43:00,2026-11-23 03:53:00,NaT,15.0,NaN,NaN
4,ORD00006,CUST0091,Mumbai,REST004,Restaurant_4,Chinese,DP006,2026-11-30 23:54:00,1284,CASH,CANCELLED,2026-11-30 23:59:00,2026-12-01 00:14:00,NaT,NaT,15.0,NaN,NaN


Rejected record sample:


,order_id,customer_id,restaurant_id,partner_id,order_timestamp,order_amount,payment_method,order_status,source_file,source_row_number,...,run_id,content_hash,validation_errors,pipeline_status,order_amount_numeric,transformation_errors,event_id,event_type,event_timestamp,event_timestamp_parsed
0,ORD00003,CUST0099,REST004,DP015,2026-11-17 14:15:00,1109,UPI,DELIVERED,orders.csv,4,...,7a5cd186-7ea5-4663-a08d-541e1c662793,7a4cccdfd06444cf12fa6d26b033ba83d7daff79be1d87...,QB-002: Duplicate order ID,REJECT,1109,,<NA>,<NA>,NaT,NaT
1,ORD00003,CUST0099,REST004,DP015,2026-11-17 14:15:00,1109,UPI,DELIVERED,orders.csv,502,...,7a5cd186-7ea5-4663-a08d-541e1c662793,7a4cccdfd06444cf12fa6d26b033ba83d7daff79be1d87...,QB-002: Duplicate order ID,REJECT,1109,,<NA>,<NA>,NaT,NaT
2,ORD90001,<NA>,REST011,DP006,2026-11-07 08:35:00,803,UPI,PREPARING,orders.csv,503,...,7a5cd186-7ea5-4663-a08d-541e1c662793,d66be8229d8f01db6f53a2f4ddec408d5dd775cff90c31...,QB-003: Customer ID missing or not found,REJECT,803,,<NA>,<NA>,NaT,NaT
3,ORD90002,CUST0086,REST999,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP,orders.csv,504,...,7a5cd186-7ea5-4663-a08d-541e1c662793,1c31f3293a2735a3228ef994777ebdb644dad99e520b09...,QB-004: Restaurant ID missing or not found,REJECT,674,,<NA>,<NA>,NaT,NaT
4,ORD90003,CUST0091,REST004,DP006,2026-11-30 23:54:00,0,CASH,CANCELLED,orders.csv,505,...,7a5cd186-7ea5-4663-a08d-541e1c662793,04b46f5be7561be09323ec95e976d1c4c91df69dd8ed46...,QB-005: Order amount must be greater than zero,REJECT,0,,<NA>,<NA>,NaT,NaT


In [ ]:

# -----------------------------------------
# QUICKBITE - PHASE 5: DATABASE PUBLISHING
# -----------------------------------------

import sqlite3
import pandas as pd
import os

DATABASE_PATH = "quickbite.db"

# Connect to SQLite database
connection = sqlite3.connect(DATABASE_PATH)

print("SQLite database connected successfully!")
print("Database:", DATABASE_PATH)


SQLite database connected successfully!
Database: quickbite.db


In [ ]:

# -----------------------------------------
# PREPARE DATABASE TABLES
# -----------------------------------------

publish_tables = {
    "customers": raw_data["customers"][
        ["customer_id", "customer_name", "email", "city"]
    ].copy(),

    "restaurants": raw_data["restaurants"][
        ["restaurant_id", "restaurant_name", "cuisine", "city"]
    ].copy(),

    "delivery_partners": raw_data["delivery_partners"][
        ["partner_id", "partner_name", "vehicle_type"]
    ].copy(),

    "trusted_orders": trusted_orders.copy(),

    "trusted_delivery_events": trusted_events.copy(),

    "delivery_performance": delivery_performance.copy(),

    "transformation_rejects": all_rejects.copy(),

    "exception_report": exception_report.copy(),

    "ingestion_report": ingestion_report.copy()
}

print("Publishing datasets prepared:")

for table_name, df in publish_tables.items():
    print(f"{table_name}: {len(df)} records")


Publishing datasets prepared:
customers: 100 records
restaurants: 15 records
delivery_partners: 20 records
trusted_orders: 499 records
trusted_delivery_events: 2004 records
delivery_performance: 499 records
transformation_rejects: 17 records
exception_report: 12 records
ingestion_report: 5 records


In [ ]:

# -----------------------------------------
# PUBLISH DATA INTO SQLITE
# -----------------------------------------

def prepare_for_sqlite(df):

    df = df.copy()

    # Convert datetime columns into ISO-formatted text
    for col in df.select_dtypes(
        include=["datetime", "datetimetz"]
    ).columns:

        df[col] = df[col].apply(
            lambda x: x.isoformat() if pd.notna(x) else None
        )

    # Replace pandas missing values with SQL NULL
    df = df.astype(object).where(pd.notna(df), None)

    # Convert any remaining Timestamp objects
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: x.isoformat()
            if isinstance(x, pd.Timestamp)
            else x
        )

    return df


publishing_results = []

for table_name, dataframe in publish_tables.items():

    try:
        sql_df = prepare_for_sqlite(dataframe)

        sql_df.to_sql(
            name=table_name,
            con=connection,
            if_exists="replace",
            index=False
        )

        publishing_results.append({
            "table_name": table_name,
            "record_count": len(dataframe),
            "publish_status": "SUCCESS"
        })

        print(
            f"SUCCESS: {table_name} | "
            f"Records: {len(dataframe)}"
        )

    except Exception as error:

        publishing_results.append({
            "table_name": table_name,
            "record_count": len(dataframe),
            "publish_status": "FAILED"
        })

        print(f"FAILED: {table_name} | {error}")

publishing_results = pd.DataFrame(publishing_results)

print("\nDATABASE PUBLISHING SUMMARY")
display(publishing_results)


SUCCESS: customers | Records: 100
SUCCESS: restaurants | Records: 15
SUCCESS: delivery_partners | Records: 20
SUCCESS: trusted_orders | Records: 499
SUCCESS: trusted_delivery_events | Records: 2004
SUCCESS: delivery_performance | Records: 499
SUCCESS: transformation_rejects | Records: 17
SUCCESS: exception_report | Records: 12
SUCCESS: ingestion_report | Records: 5

DATABASE PUBLISHING SUMMARY


,table_name,record_count,publish_status
0,customers,100,SUCCESS
1,restaurants,15,SUCCESS
2,delivery_partners,20,SUCCESS
3,trusted_orders,499,SUCCESS
4,trusted_delivery_events,2004,SUCCESS
5,delivery_performance,499,SUCCESS
6,transformation_rejects,17,SUCCESS
7,exception_report,12,SUCCESS
8,ingestion_report,5,SUCCESS


In [ ]:

# -----------------------------------------
# VERIFY SQLITE TABLES
# -----------------------------------------

cursor = connection.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""")

table_names = [row[0] for row in cursor.fetchall()]

print("TABLES AVAILABLE IN QUICKBITE DATABASE")

for table in table_names:
    print("-", table)

# Check row counts directly in SQLite
verification_results = []

for table in table_names:

    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS total FROM "{table}"',
        connection
    ).iloc[0, 0]

    verification_results.append({
        "table_name": table,
        "database_row_count": count
    })

verification_report = pd.DataFrame(verification_results)

print("\nDATABASE VERIFICATION REPORT")
display(verification_report)


TABLES AVAILABLE IN QUICKBITE DATABASE
- customers
- delivery_partners
- delivery_performance
- exception_report
- ingestion_report
- restaurants
- transformation_rejects
- trusted_delivery_events
- trusted_orders

DATABASE VERIFICATION REPORT


,table_name,database_row_count
0,customers,100
1,delivery_partners,20
2,delivery_performance,499
3,exception_report,12
4,ingestion_report,5
5,restaurants,15
6,transformation_rejects,17
7,trusted_delivery_events,2004
8,trusted_orders,499


In [ ]:

# Execute SQL queries using Pandas

query = """
SELECT
    restaurant_name,
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue
FROM delivery_performance
GROUP BY restaurant_name
ORDER BY total_orders DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, connection)

display(result)



,restaurant_name,total_orders,total_revenue
0,Restaurant_5,41,32024.0
1,Restaurant_15,40,34424.0
2,Restaurant_6,38,28359.0
3,Restaurant_10,38,30811.0
4,Restaurant_13,36,31383.0
5,Restaurant_1,36,30736.0
6,Restaurant_12,34,23744.0
7,Restaurant_9,33,26999.0
8,Restaurant_3,33,24815.0
9,Restaurant_2,31,22304.0


In [ ]:

connection.close()

print("SQLite database connection closed.")


SQLite database connection closed.


In [ ]:

from google.colab import files

files.download("quickbite.db")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("quickbite.db")

print("Connected to QuickBite database successfully!")

Connected to QuickBite database successfully!


In [ ]:
# List all tables
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", connection)

display(tables)

# Show the columns in each table
for table_name in tables["name"]:
    print(f"\n--- {table_name} ---")

    schema = pd.read_sql_query(
        f"PRAGMA table_info('{table_name}')",
        connection
    )

    display(schema[["name", "type"]])

,name
0,customers
1,delivery_partners
2,delivery_performance
3,exception_report
4,ingestion_report
5,restaurants
6,transformation_rejects
7,trusted_delivery_events
8,trusted_orders



--- customers ---


,name,type
0,customer_id,TEXT
1,customer_name,TEXT
2,email,TEXT
3,city,TEXT



--- delivery_partners ---


,name,type
0,partner_id,TEXT
1,partner_name,TEXT
2,vehicle_type,TEXT



--- delivery_performance ---


... (9 outputs left out)


--- trusted_delivery_events ---


,name,type
0,event_id,TEXT
1,order_id,TEXT
2,event_type,TEXT
3,event_timestamp,TEXT
4,source_file,TEXT
5,source_row_number,INTEGER
6,loaded_at,TEXT
7,run_id,TEXT
8,content_hash,TEXT
9,validation_errors,TEXT



--- trusted_orders ---


,name,type
0,order_id,TEXT
1,customer_id,TEXT
2,restaurant_id,TEXT
3,partner_id,TEXT
4,order_timestamp,TEXT
5,order_amount,INTEGER
6,payment_method,TEXT
7,order_status,TEXT
8,source_file,TEXT
9,source_row_number,INTEGER


In [ ]:
query = """
SELECT
    COUNT(*) AS total_trusted_orders
FROM trusted_orders;
"""

result = pd.read_sql_query(query, connection)
display(result)

,total_trusted_orders
0,499


In [ ]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM trusted_orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

status_analysis = pd.read_sql_query(query, connection)
display(status_analysis)

,order_status,total_orders
0,DELIVERED,203
1,READY_FOR_PICKUP,80
2,PREPARING,63
3,PICKED_UP,53
4,PLACED,50
5,CANCELLED,50


In [ ]:
query = """
SELECT
    payment_method,
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue,
    ROUND(AVG(order_amount), 2) AS average_order_value
FROM trusted_orders
GROUP BY payment_method
ORDER BY total_revenue DESC;
"""

revenue_by_payment = pd.read_sql_query(query, connection)
display(revenue_by_payment)

,payment_method,total_orders,total_revenue,average_order_value
0,CARD,135,112169.0,830.88
1,WALLET,132,103751.0,785.99
2,UPI,120,93173.0,776.44
3,CASH,112,88311.0,788.49


In [ ]:
query = """
SELECT
    restaurant_name,
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue,
    ROUND(AVG(order_amount), 2) AS average_order_value
FROM delivery_performance
GROUP BY restaurant_name
ORDER BY total_revenue DESC
LIMIT 10;
"""

restaurant_performance = pd.read_sql_query(query, connection)
display(restaurant_performance)

,restaurant_name,total_orders,total_revenue,average_order_value
0,Restaurant_15,40,34424.0,860.60
1,Restaurant_5,41,32024.0,781.07
2,Restaurant_13,36,31383.0,871.75
3,Restaurant_10,38,30811.0,810.82
4,Restaurant_1,36,30736.0,853.78
5,Restaurant_6,38,28359.0,746.29
6,Restaurant_9,33,26999.0,818.15
7,Restaurant_3,33,24815.0,751.97
8,Restaurant_14,29,24023.0,828.38
9,Restaurant_7,28,23987.0,856.68


In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    ROUND(AVG(preparation_minutes), 2) AS avg_preparation_minutes,
    ROUND(AVG(delivery_minutes), 2) AS avg_delivery_minutes,
    ROUND(AVG(total_fulfillment_minutes), 2) AS avg_total_minutes
FROM delivery_performance;
"""

delivery_analysis = pd.read_sql_query(query, connection)
display(delivery_analysis)

,total_orders,avg_preparation_minutes,avg_delivery_minutes,avg_total_minutes
0,499,15.0,25.0,55.0


In [ ]:
query = """
SELECT
    SUBSTR(validation_errors, 1, 6) AS rule_id,
    COUNT(*) AS exception_count
FROM exception_report
WHERE validation_errors IS NOT NULL
  AND validation_errors != ''
GROUP BY rule_id
ORDER BY exception_count DESC;
"""

quality_analysis = pd.read_sql_query(query, connection)
display(quality_analysis)

,rule_id,exception_count
0,QB-008,2
1,QB-002,2
2,QB-012,1
3,QB-010,1
4,QB-009,1
5,QB-007,1
6,QB-006,1
7,QB-005,1
8,QB-004,1
9,QB-003,1


In [ ]:
# Display all columns in exception_report
query = "PRAGMA table_info(exception_report);"

exception_schema = pd.read_sql_query(query, connection)
display(exception_schema)

# Display the first five rows
display(pd.read_sql_query(
    "SELECT * FROM exception_report LIMIT 5;",
    connection
))

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,restaurant_id,TEXT,0,None,0
3,3,partner_id,TEXT,0,None,0
4,4,order_timestamp,TEXT,0,None,0
5,5,order_amount,TEXT,0,None,0
6,6,payment_method,TEXT,0,None,0
7,7,order_status,TEXT,0,None,0
8,8,source_file,TEXT,0,None,0
9,9,source_row_number,INTEGER,0,None,0


,order_id,customer_id,restaurant_id,partner_id,order_timestamp,order_amount,payment_method,order_status,source_file,source_row_number,...,run_id,content_hash,validation_errors,pipeline_status,order_amount_numeric,source_dataset,event_id,event_type,event_timestamp,event_timestamp_parsed
0,ORD00003,CUST0099,REST004,DP015,2026-11-17 14:15:00,1109,UPI,DELIVERED,orders.csv,4,...,7a5cd186-7ea5-4663-a08d-541e1c662793,7a4cccdfd06444cf12fa6d26b033ba83d7daff79be1d87...,QB-002: Duplicate order ID,REJECT,1109.0,orders,None,None,None,None
1,ORD00003,CUST0099,REST004,DP015,2026-11-17 14:15:00,1109,UPI,DELIVERED,orders.csv,502,...,7a5cd186-7ea5-4663-a08d-541e1c662793,7a4cccdfd06444cf12fa6d26b033ba83d7daff79be1d87...,QB-002: Duplicate order ID,REJECT,1109.0,orders,None,None,None,None
2,ORD90001,None,REST011,DP006,2026-11-07 08:35:00,803,UPI,PREPARING,orders.csv,503,...,7a5cd186-7ea5-4663-a08d-541e1c662793,d66be8229d8f01db6f53a2f4ddec408d5dd775cff90c31...,QB-003: Customer ID missing or not found,REJECT,803.0,orders,None,None,None,None
3,ORD90002,CUST0086,REST999,DP017,2026-11-23 03:23:00,674,CARD,PICKED_UP,orders.csv,504,...,7a5cd186-7ea5-4663-a08d-541e1c662793,1c31f3293a2735a3228ef994777ebdb644dad99e520b09...,QB-004: Restaurant ID missing or not found,REJECT,674.0,orders,None,None,None,None
4,ORD90003,CUST0091,REST004,DP006,2026-11-30 23:54:00,0,CASH,CANCELLED,orders.csv,505,...,7a5cd186-7ea5-4663-a08d-541e1c662793,04b46f5be7561be09323ec95e976d1c4c91df69dd8ed46...,QB-005: Order amount must be greater than zero,REJECT,0.0,orders,None,None,None,None


In [ ]:
query = """
SELECT
    validation_errors,
    pipeline_status,
    COUNT(*) AS total_records
FROM exception_report
WHERE validation_errors IS NOT NULL
  AND validation_errors != ''
GROUP BY validation_errors, pipeline_status
ORDER BY total_records DESC;
"""

display(pd.read_sql_query(query, connection))

,validation_errors,pipeline_status,total_records
0,QB-002: Duplicate order ID,REJECT,2
1,QB-008: Duplicate event ID,REJECT,2
2,QB-003: Customer ID missing or not found,REJECT,1
3,QB-004: Restaurant ID missing or not found,REJECT,1
4,QB-005: Order amount must be greater than zero,REJECT,1
5,QB-006: Invalid order status,REJECT,1
6,QB-007: Partner ID missing or not found,REJECT,1
7,QB-009: Missing or unknown order ID,REJECT,1
8,QB-010: Delivery occurred before pickup,HUMAN_REVIEW,1
9,QB-012: Cancelled order has a delivered event,HUMAN_REVIEW,1


In [ ]:
query = """
SELECT
    DATE(order_timestamp) AS order_date,
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS daily_revenue
FROM trusted_orders
GROUP BY DATE(order_timestamp)
ORDER BY order_date;
"""

daily_trend = pd.read_sql_query(query, connection)
display(daily_trend)

,order_date,total_orders,daily_revenue
0,2026-11-01,18,14910.0
1,2026-11-02,15,11476.0
2,2026-11-03,14,15017.0
3,2026-11-04,14,12978.0
4,2026-11-05,16,10531.0
5,2026-11-06,12,10390.0
6,2026-11-07,19,12333.0
7,2026-11-08,16,13115.0
8,2026-11-09,17,13222.0
9,2026-11-10,18,15409.0


In [ ]:
query = """
SELECT
    partner_id,
    COUNT(*) AS completed_deliveries,
    ROUND(AVG(delivery_minutes), 2) AS avg_delivery_minutes
FROM delivery_performance
WHERE delivery_minutes IS NOT NULL
GROUP BY partner_id
ORDER BY completed_deliveries DESC;
"""

partner_performance = pd.read_sql_query(query, connection)
display(partner_performance)

,partner_id,completed_deliveries,avg_delivery_minutes
0,DP010,13,25.0
1,DP008,13,25.0
2,DP005,13,25.0
3,DP019,12,25.0
4,DP012,12,25.0
5,DP002,12,25.0
6,DP001,12,25.0
7,DP017,11,25.0
8,DP015,11,25.0
9,DP014,11,25.0
